# Кольцевые коммуникации

Реализуйте три операции на CPU с Gloo. 
- `rank` — номер процесса,
- `world_size` — число процессов.
- Входные тензоры — одномерные CPU-тензоры одинакового размера на всех процессах. Их менять нельзя

In [1]:
%%writefile collectives.py
import torch
import torch.distributed as dist


def ring_reduce_scatter(tensor):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    if tensor.ndim != 1 or tensor.numel() % world_size:
        raise ValueError("Expected a 1D tensor with length divisible by world_size")
    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]
    previous_rank = (rank - 1) % world_size
    next_rank = (rank + 1) % world_size

    # The last received and summed chunk must belong to this rank.
    for step in range(world_size - 1):
        send_index = (rank - step - 1) % world_size
        receive_index = (rank - step - 2) % world_size
        received = torch.empty_like(chunks[receive_index])
        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], next_rank),
            dist.P2POp(dist.irecv, received, previous_rank),
        ])
        for request in requests:
            request.wait()
        chunks[receive_index].add_(received)
    return chunks[rank]


def ring_all_gather(shard):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)
    previous_rank = (rank - 1) % world_size
    next_rank = (rank + 1) % world_size

    for step in range(world_size - 1):
        send_index = (rank - step) % world_size
        receive_index = (rank - step - 1) % world_size
        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], next_rank),
            dist.P2POp(dist.irecv, chunks[receive_index], previous_rank),
        ])
        for request in requests:
            request.wait()
    return torch.cat(chunks)


def ring_all_reduce(tensor):
    return ring_all_gather(ring_reduce_scatter(tensor))


Overwriting collectives.py


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [2]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Overwriting check.py


In [3]:
import socket
import subprocess
import sys


def check_ring(world_size):
    # Explicit loopback also works when the laptop's hostname is not in DNS.
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        port = sock.getsockname()[1]
    subprocess.run(
        [sys.executable, "-m", "torch.distributed.run", "--nnodes=1",
         "--master-addr=127.0.0.1", f"--master-port={port}",
         f"--nproc-per-node={world_size}", "check.py"],
        check=True,
    )


check_ring(2)


W1014 14:13:42.459000 15122 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


[W1014 14:13:43.328108000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1014 14:13:43.328123000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


OK: все три операции прошли проверки на 2 процессах.


In [4]:
check_ring(4)


W1014 14:13:44.707000 15132 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


[W1014 14:13:45.809276000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1014 14:13:45.809316000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1014 14:13:45.809340000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1014 14:13:45.809349000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


OK: все три операции прошли проверки на 4 процессах.


Дополнительно проверим один процесс и нечётное число процессов: эти случаи проверяют отсутствие обменов и замыкание кольца.

In [5]:
check_ring(1)
check_ring(3)


W1014 14:13:47.182000 15165 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


[W1014 14:13:48.972680000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


OK: все три операции прошли проверки на 1 процессах.


W1014 14:13:49.377000 15180 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


[W1014 14:13:50.245855000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1014 14:13:50.245885000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W1014 14:13:50.245891000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


OK: все три операции прошли проверки на 3 процессах.
